# GW150914 — pycWB Pure-Python Tutorial

End-to-end analysis of GW150914 using the pure-Python wrappers:
`coherence` → `supercluster_wrapper` → `likelihood_wrapper`

## 0. Install dependencies

In [ ]:
!pip install -q pycwb gwosc gwpy requests healpy

## 1. Configuration

In [ ]:
import os
import pycwb
from pycwb.config import Config
from pycwb.modules.logger import logger_init

if not os.environ.get('HOME_WAT_FILTERS'):
    pyburst_path = os.path.dirname(os.path.abspath(pycwb.__file__))
    os.environ['HOME_WAT_FILTERS'] = f"{os.path.abspath(pyburst_path)}/vendor"

logger_init()

In [ ]:
user_parameters = """
analysis: "2G"
cfg_search: "r"
optim: False

ifo: ["L1","H1"]
refIFO: "L1"

inRate: 16384

lagSize: 1
lagStep: 1.
lagOff: 0
lagMax: 0

slagSize: 0
slagMin: 0
slagMax: 0
slagOff: 0

segLen: 1200
segMLS: 600
segTHR: 200
segEdge: 10

fLow: 16.
fHigh: 1024.

levelR: 3
l_low: 4
l_high: 10

wdmXTalk: "wdmXTalk/OverlapCatalog16-1024.bin"

healpix: 7

bpp: 0.001
subnet: 0.5
subcut: 0.0
netRHO: 5.5
netCC: 0.5
Acore: 1.7
Tgap: 0.2
Fgap: 128.0
delta: 0.5
cfg_gamma: -1.0
LOUD: 300

pattern: 5
iwindow: 30
nSky: 196608
nfactor: 1
"""

with open('user_parameters.yaml', 'w') as fp:
    fp.write(user_parameters)

config = Config()
config.load_from_yaml('user_parameters.yaml')
print("Config loaded:", config.ifo)

## 2. Download GW150914 open data

In [ ]:
import requests
from gwpy.timeseries import TimeSeries
from gwosc.locate import get_urls

t0 = 1126259462.4
half_win = 150  # seconds each side of t0 to load
data = []

for ifo in config.ifo:
    url = get_urls(ifo, t0, t0, sample_rate=config.inRate)[-1]
    print(f"Downloading {ifo}: {url}")
    fn = os.path.basename(url)
    # skip download if file already exists
    if not os.path.exists(fn):
        with open(fn, 'wb') as f:
            f.write(requests.get(url).content)
    strain = TimeSeries.read(fn, format='hdf5.gwosc')
    data.append(strain.crop(t0 - half_win, t0 + half_win))
    print(f"  {ifo}: {len(data[-1])} samples @ {data[-1].sample_rate}")

# define a job segment for this search
from pycwb.types.job import WaveSegment
job_segment = WaveSegment(
    index=0,
    ifos=config.ifo,
    analyze_start=t0 - half_win + config.segEdge,
    analyze_end=t0 + half_win - config.segEdge,
    sample_rate=config.rateANA,
    seg_edge=config.segEdge,
    lag_size=config.lagSize,
    lag_step=config.lagStep,
    lag_off=config.lagOff,
    lag_max=config.lagMax,
    shift=[0,0]
)
print(f"Job segment: [{job_segment.analyze_start}, {job_segment.analyze_end}]  "
      f"duration={job_segment.duration}s  n_lag={job_segment.n_lag}")

## 3. Data conditioning

In [ ]:
from pycwb.modules.read_data.data_check import check_and_resample_py
from pycwb.modules.data_conditioning.data_conditioning_python import data_conditioning

data = [check_and_resample_py(data[i], config, i) for i in range(len(config.ifo))]
strains, nRMS = data_conditioning(config, data)
print(f"Conditioned {len(strains)} strain(s), {len(nRMS)} nRMS map(s)")

In [ ]:
%matplotlib inline
from pycwb.modules.plot import plot_spectrogram

ax = plot_spectrogram(strains[0], gwpy_plot=True).gca()
ax.set_ylim(15, 1024)
ax.set_title("L1 conditioned strain")

## 4. Coherence

Pixel selection and fragment clustering across all WDM resolutions.

In [ ]:
from pycwb.modules.cwb_coherence import coherence

fragment_clusters = coherence(config, strains)
n_res = len(fragment_clusters)
n_clusters = sum(len(fragment_clusters[r][0].clusters) for r in range(n_res))
print(f"Resolutions: {n_res}  |  total fragment clusters (lag 0): {n_clusters}")

## 5. Supercluster

TD amplitude extraction, multi-resolution merging, and subnet veto.

In [ ]:
from pycwb.modules.super_cluster import supercluster_wrapper
from pycwb.modules.xtalk.monster import load_catalog

xtalk_coeff, xtalk_lookup_table, layers, _ = load_catalog(config.MRAcatalog)

super_fragment_clusters = supercluster_wrapper(
    config, fragment_clusters, strains,
    xtalk_coeff, xtalk_lookup_table, layers,
)

if super_fragment_clusters is None:
    print("No clusters survived the subnet cut")
else:
    total = sum(len(fc.clusters) for fc in super_fragment_clusters)
    print(f"Lags: {len(super_fragment_clusters)}  |  total superclusters: {total}")

## 6. Likelihood

Sky scan and reconstruction veto across all surviving clusters.

In [ ]:
from pycwb.modules.likelihoodWP import likelihood_wrapper

results = likelihood_wrapper(
    config, super_fragment_clusters, strains,
    config.MRAcatalog, nRMS=nRMS,
)

total_accepted = sum(len(r) for r in results)
print(f"Accepted clusters: {total_accepted}")
for lag, lag_results in enumerate(results):
    for i, (cluster, sky_stats) in enumerate(lag_results):
        print(f"  lag={lag}  cluster={i+1}  pixels={len(cluster.pixel_arrays)}"
              f"  rho={cluster.cluster_meta.net_rho:.2f}"
              f"  cc={cluster.cluster_meta.net_cc:.2f}")

## 7. Results

In [ ]:
%matplotlib inline
from pycwb.modules.plot import plot_event_on_spectrogram
from pycwb.types.network_event import Event

events = []
for lag_results in results:
    for result_cluster, sky_stats in lag_results:
        event = Event()
        event.output_py(job_segment, result_cluster, config)
        events.append(event)

if events:
    plot_event_on_spectrogram(strains[0], events).show()
else:
    print("No accepted events to plot")

In [ ]:
%matplotlib inline
from gwpy.spectrogram import Spectrogram

for lag_results in results:
    for result_cluster, sky_stats in lag_results:
        merged_map, start, dt, df = result_cluster.get_sparse_map("likelihood")
        ax = Spectrogram(merged_map, t0=start, dt=dt, f0=0, df=df).plot().gca()
        ax.set_title(f"Likelihood map  rho={result_cluster.cluster_meta.net_rho:.2f}")

In [ ]:
%matplotlib inline
from pycwb.modules.reconstruction import get_network_MRA_wave
from matplotlib import pyplot as plt

for lag_results in results:
    for result_cluster, sky_stats in lag_results:
        waves = get_network_MRA_wave(
            config, result_cluster, config.rateANA, config.nIFO, config.TDRate,
            'signal', 0, True,
        )
        fig, ax = plt.subplots()
        for wave in waves:
            ax.plot(wave.sample_times, wave.data)
        ax.set_xlabel("GPS time (s)")
        ax.set_ylabel("Strain")
        ax.set_title("Reconstructed waveform")
        plt.show()